In [6]:
import requests
import pandas as pd
import time
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

ACCESS_TOKEN = "EAAgjgrrdkIUBSzYe7NAADDFpqsQoTAOuPRZAEkZAZCB2MSlGCxxGLdCdpf37wGdlCDkfwhJwwMiQOWZAmZCMckc6kA4vgapdwSAZCt6CHYydzZCrnMLPUpe3tFg2uoxZCBlSSjU0wn45ij8vfAXC1wIpLcbN2I0RdqYdXiLrSVTQGKoZAQXD5BvHN5nHsEcTOAxRrl3HqN7HalSxJrLUm0ktYWVohvHuisMs0ZChMhvrpNceNPEZAFl5yZB2ZCuVSJeZCAnInGCX3x6ZCR7Mgfb"
IG_USER_ID = "17841480144439476" ## 내 아이디
HASHTAG_ID = "17843691760058549" ## 검색 해시태그 이이디

# Avf1NY7r5_ZN4H3pQJ9ws2_
# ADBIc9DLNjcvpVP1pnhIeEl

TARGET_COUNT = 3000
PAGE_LIMIT = 20
MAX_PAGES = 250

url = f"https://graph.facebook.com/v26.0/{HASHTAG_ID}/top_media"

params = {
    "user_id": IG_USER_ID,
    "fields": "id,caption,media_type,permalink,timestamp,like_count,comments_count",
    "limit": PAGE_LIMIT,
    "access_token": ACCESS_TOKEN
}

# =====================================
# 2. 데이터 저장 변수
# =====================================

all_posts = []
seen_ids = set()

page_num = 0
no_new_pages = 0

# =====================================
# 3. 인스타그램 게시글 수집
# =====================================

while len(all_posts) < TARGET_COUNT and page_num < MAX_PAGES:

    try:

        # ---------------------------------
        # API 요청 및 오류 재시도
        # ---------------------------------

        success = False

        for retry in range(3):

            response = requests.get(
                url,
                params=params,
                timeout=60
            )

            result = response.json()

            if response.status_code == 200 and "error" not in result:
                success = True
                break

            print(
                f"API 오류 ({retry + 1}/3):",
                result.get("error", result)
            )

            # 인증/권한 오류는 재시도하지 않음
            error_code = result.get("error", {}).get("code")

            if error_code in [104, 190, 200]:
                print("인증 또는 권한 오류로 중단")
                break

            if retry < 2:
                wait_time = 10 * (2 ** retry)

                print(f"{wait_time}초 후 재시도")
                time.sleep(wait_time)

        if not success:
            print("API 요청 실패로 수집 중단")
            break

        # ---------------------------------
        # 게시글 데이터 처리
        # ---------------------------------

        posts = result.get("data", [])

        page_num += 1
        new_count = 0

        for post in posts:

            post_id = post.get("id")

            if post_id and post_id not in seen_ids:

                seen_ids.add(post_id)
                all_posts.append(post)
                new_count += 1

            if len(all_posts) >= TARGET_COUNT:
                break

        # ---------------------------------
        # 수집 현황 출력
        # ---------------------------------

        print(f"\n{page_num}페이지")
        print(f"응답 건수: {len(posts)}")
        print(f"신규 수집: {new_count}")
        print(f"누적 수집: {len(all_posts)}")

        # ---------------------------------
        # 목표 수집량 확인
        # ---------------------------------

        if len(all_posts) >= TARGET_COUNT:
            print("목표 수집량 달성")
            break

        # ---------------------------------
        # 신규 게시글 여부 확인
        # ---------------------------------

        if new_count == 0:
            no_new_pages += 1
        else:
            no_new_pages = 0

        if no_new_pages >= 2:
            print("신규 게시글이 2페이지 연속 없어 종료")
            break

        # ---------------------------------
        # 다음 페이지 확인
        # ---------------------------------

        paging = result.get("paging", {})
        next_url = paging.get("next")

        if not next_url:
            print("다음 페이지가 없어 종료")
            break

        # ---------------------------------
        # 다음 페이지 URL 및 토큰 유지
        # ---------------------------------

        parsed = urlparse(next_url)
        query_params = parse_qs(parsed.query)

        query_params["access_token"] = [ACCESS_TOKEN]

        url = urlunparse(
            parsed._replace(
                query=urlencode(
                    query_params,
                    doseq=True
                )
            )
        )

        params = None

        # ---------------------------------
        # API 요청 간격
        # ---------------------------------

        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print("네트워크 오류:", e)
        break

    except ValueError as e:
        print("응답 처리 오류:", e)
        break

# =====================================
# 4. CSV 저장
# =====================================

df = pd.DataFrame(
    all_posts,
    columns=[
        "id",
        "caption",
        "media_type",
        "permalink",
        "timestamp",
        "like_count",
        "comments_count"
    ]
)

df.to_csv(
    "instagram_쿠리만쥬_final_02.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 5. 최종 결과 출력
# =====================================

print("\n==========================")
print("인스타그램 데이터 수집 종료")
print("==========================")

print(f"목표 수집량: {TARGET_COUNT}건")
print(f"실제 수집량: {len(df)}건")
print(f"요청 페이지: {page_num}개")
print("저장 파일: instagram_쿠리만쥬_final_02.csv")



1페이지
응답 건수: 20
신규 수집: 20
누적 수집: 20

2페이지
응답 건수: 20
신규 수집: 20
누적 수집: 40

3페이지
응답 건수: 20
신규 수집: 20
누적 수집: 60

4페이지
응답 건수: 20
신규 수집: 20
누적 수집: 80

5페이지
응답 건수: 20
신규 수집: 20
누적 수집: 100

6페이지
응답 건수: 20
신규 수집: 20
누적 수집: 120

7페이지
응답 건수: 20
신규 수집: 20
누적 수집: 140

8페이지
응답 건수: 20
신규 수집: 20
누적 수집: 160

9페이지
응답 건수: 20
신규 수집: 20
누적 수집: 180

10페이지
응답 건수: 20
신규 수집: 20
누적 수집: 200

11페이지
응답 건수: 20
신규 수집: 20
누적 수집: 220

12페이지
응답 건수: 20
신규 수집: 20
누적 수집: 240

13페이지
응답 건수: 20
신규 수집: 20
누적 수집: 260

14페이지
응답 건수: 20
신규 수집: 20
누적 수집: 280

15페이지
응답 건수: 20
신규 수집: 20
누적 수집: 300

16페이지
응답 건수: 20
신규 수집: 20
누적 수집: 320

17페이지
응답 건수: 20
신규 수집: 20
누적 수집: 340

18페이지
응답 건수: 20
신규 수집: 20
누적 수집: 360

19페이지
응답 건수: 19
신규 수집: 19
누적 수집: 379

20페이지
응답 건수: 20
신규 수집: 20
누적 수집: 399

21페이지
응답 건수: 20
신규 수집: 20
누적 수집: 419

22페이지
응답 건수: 20
신규 수집: 20
누적 수집: 439

23페이지
응답 건수: 20
신규 수집: 20
누적 수집: 459

24페이지
응답 건수: 20
신규 수집: 20
누적 수집: 479

25페이지
응답 건수: 19
신규 수집: 19
누적 수집: 498

26페이지
응답 건수: 19
신규 수집: 19
누적 수집: 517

27페이지
응답 건수: 19
신규 수집: 1